# as-strided-windowing — ex2: batched + channelled windowing for conv1d input prep

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `as-strided-windowing`. Running the final beacon cell reports progress against the `PyTorch: as_strided windowing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: as_strided windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-windowing`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-windowing"
DD_SUBTOPIC = "PyTorch: as_strided windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `as_strided` windowing — quick refresher

`t.as_strided(x, size, stride)` builds a **zero-copy** view at exactly the `(size, stride)` you specify. The classic sliding-window trick is to set both the window-position stride *and* the within-window stride to the source's element stride, so the view becomes a (n_windows, window_size) matrix that aliases the source.

**Pattern for 1-D sliding window of width `K` over a 1-D tensor `x` of length `L`:**
```python
sL, = x.stride()
windows = t.as_strided(x, size=(L - K + 1, K), stride=(sL, sL))
```

**Generalises to N-D inputs.** For batched/channelled input `(B, IC, W)`, you pull all of `x.stride()` and build a `(B, IC, L_out, K)` view with stride `(sB, sIC, sW, sW)`. This is exactly the ARENA `conv1d_minimal` trick.

**Why you need the source's stride, not `1`.** If `x` was itself created via `permute`/`transpose`/another `as_strided` call, its last stride may be larger than 1. Hard-coding `1` will silently scan the wrong memory cells.

### Exercise 2 — batched + channelled windowing for conv1d input prep

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Build the (B, IC, L_out, K) windowed view of a (B, IC, W) input used as the windowed-input matrix in ARENA's `conv1d_minimal` — without performing the convolution itself.
> Keywords: conv1d, batch-channel-windowing, as_strided
> ```

**KCs targeted:** `as-strided-window-stride`, `conv1d-input-prep-shape`

Implement `ex2_conv1d_windows(x, kernel_width)`. Given an input of shape `(B, IC, W)` and an integer `kernel_width = K`, return a zero-copy view of shape `(B, IC, L_out, K)` where `L_out = W - K + 1`. Each `(L_out, K)` slice is the sliding-window matrix for one (batch, channel) pair.

**Strides to use.** Pull all of `x.stride()` (call it `s_B, s_IC, s_W`) and build the new view with stride `(s_B, s_IC, s_W, s_W)`. The last two axes both advance one spatial step.

Inputs:
- `x`: `(B, IC, W)` float tensor.
- `kernel_width`: int, `<= W`.

Output: zero-copy view of shape `(B, IC, L_out, K)`.

**Don't run the convolution** — this drill exercises ONLY the input-prep step. ARENA's `conv1d_minimal` then einsums this view with the kernel; that's a separate atom.

In [ ]:
def ex2_conv1d_windows(x: Tensor, kernel_width: int) -> Tensor:
    """Build the (B, IC, L_out, K) windowed view used in conv1d input prep."""
    raise NotImplementedError()


def _test_ex2():
    B, IC, W, K = 2, 3, 7, 3
    x = t.arange(B * IC * W, dtype=t.float32).reshape(B, IC, W)
    win = ex2_conv1d_windows(x, kernel_width=K)
    L_out = W - K + 1
    assert win.shape == (B, IC, L_out, K), (
        f'expected {(B, IC, L_out, K)}, got {tuple(win.shape)}'
    )
    assert win.dtype == t.float32
    # Must be a zero-copy view — shares storage with x.
    assert win.data_ptr() == x.data_ptr(), 'windows view must alias x storage'

    # Spot-check a window.
    # For batch 0, channel 1, window 0, the K elements are x[0, 1, 0:K].
    assert t.equal(win[0, 1, 0], x[0, 1, 0:K]), 'window[0,1,0] mismatch'
    # Last window for batch 1, channel 2: x[1, 2, W-K:W].
    assert t.equal(win[1, 2, -1], x[1, 2, W - K:W]), (
        f'window[1,2,-1] mismatch: got {win[1, 2, -1]}, expected {x[1, 2, W - K:W]}'
    )

    # Cross-check by running the conv1d pipeline end-to-end.
    import torch.nn.functional as F
    rng = t.Generator().manual_seed(1)
    x_big = t.randn(2, 3, 16, generator=rng)
    weights = t.randn(5, 3, 4, generator=rng)  # (OC, IC, K)
    win_big = ex2_conv1d_windows(x_big, kernel_width=4)
    assert win_big.shape == (2, 3, 13, 4)
    # Apply the einsum from ARENA's conv1d_minimal and compare to F.conv1d.
    ours = t.einsum('bicwk,ocik->bocw'.replace('cw', 'lk').replace('ic', 'i'), win_big, weights) if False else \
           t.einsum('b i l k, o i k -> b o l', win_big, weights)
    ref = F.conv1d(x_big, weights)
    assert t.allclose(ours, ref, atol=1e-4), (
        f'conv1d via your windows + einsum doesn\'t match F.conv1d:\n'
        f'max diff = {(ours - ref).abs().max().item()}'
    )
    print(f'window view shape {tuple(win.shape)} verified zero-copy + matches F.conv1d when combined with einsum')
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_conv1d_windows(x: Tensor, kernel_width: int) -> Tensor:
    B, IC, W = x.shape
    s_B, s_IC, s_W = x.stride()
    L_out = W - kernel_width + 1
    return t.as_strided(
        x,
        size=(B, IC, L_out, kernel_width),
        stride=(s_B, s_IC, s_W, s_W),
    )
```

**This is the load-bearing trick of ARENA's `conv1d_minimal`.** Once you have the `(B, IC, L_out, K)` view, the actual convolution is a one-line einsum: `einsum('b i l k, o i k -> b o l', windows, weights)`. The windowing IS the hard part; the contraction is just a tensor multiply.

**Always pull stride from `x.stride()`.** ARENA's source code has an explicit comment warning that hard-coding the last stride to `1` is the #1 silent-bug pattern when this trick is extended to conv2d (where the last stride for a non-contiguous channel-major input would be wrong).

**Memory cost is zero, but the view IS overlapping.** Adjacent windows share `K - 1` elements. That's fine for read-only use (einsum) but you must NOT write through this view — writes to overlapping memory have undefined order.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()